In [0]:
from pyspark.sql.functions import current_timestamp, expr, window, col
from pyspark.sql.types import *

# Simulate events arriving with timestamps — some arrive late
events_df = (
    spark.readStream
    .format("rate")
    .option("rowsPerSecond", 2)
    .load()
    # Simulate some events being "late" by subtracting random seconds
    .withColumn("event_time", current_timestamp() - expr("INTERVAL 30 SECONDS * rand()"))
    .withColumn("user_id", (col("value") % 5).cast("int"))
    .withColumn("amount", (col("value") % 100).cast("double"))
)

events_df.printSchema()

In [0]:
import time

# Watermark tells Spark: "wait up to 1 minute for late data,
# then discard anything older than that"

windowed = (
    events_df
    .withWatermark("event_time", "1 minute")   # ← key line
    .groupBy(
        window(col("event_time"), "30 seconds"),  # 30-second tumbling window
        col("user_id")
    )
    .agg({"amount": "sum"})
    .withColumnRenamed("sum(amount)", "total_amount")
)

# Write aggregated results to Delta
agg_query = (
    windowed.writeStream
    .format("delta")
    .option("checkpointLocation", "/Volumes/workspace/default/streaming_demo/checkpoints/windowed")
    .outputMode("append")   # use "append" with watermark
    .trigger(availableNow=True)
    .start("/Volumes/workspace/default/streaming_demo/delta/windowed_agg")
)

agg_query.awaitTermination()

spark.read.format("delta").load("/Volumes/workspace/default/streaming_demo/delta/windowed_agg").show()